Purpose: To group the location of the housing based on island etc.

API: https://psgc.gitlab.io/api/


Getting the data of the following:
- Island 
- Province
- Cities 

Output: 
- Type: CSV
- Tables: 
    - Province
    -  Cities


In [0]:
import requests
import pandas as pd
from pyspark.sql import functions as F     

BASE_URL = 'https://psgc.gitlab.io/api/'


def fetch_data(endpoint: str) -> pd.DataFrame:
    url = f"{BASE_URL}/{endpoint}"
    response = requests.get(url, timeout=30)
    # Check the request status
    response.raise_for_status() 
    return pd.DataFrame( response.json())


In [0]:
## Pull data from API
provinces_pd = fetch_data('provinces')
cities_pd = fetch_data('cities')

# display( provinces_pd)
## Create spark dataframes
# regions = spark.createDataFrame(regions_pd)
# provinces')



In [0]:

# Use False if there is no municipalities or district code 
column_to_normalize = ['provinceCode', 'districtCode']
df_to_normalize = [cities_pd]

for df in df_to_normalize:
    if (df[column_to_normalize] == False).any().any():
        print('Found false value')
    df[column_to_normalize] = df[column_to_normalize].replace({False: None})

In [0]:
# Convert to Spark DataFrames
province_df = spark.createDataFrame(provinces_pd)
city_df =  spark.createDataFrame(cities_pd)

location_bronze = 'abfss://bronze@asterisktotle01.dfs.core.windows.net/PHHousing/'

# Write on Bronze layer (raw API)
province_df.write.format("delta").mode("overwrite").save(f"{location_bronze}/psgc_provinces")
city_df.write.format("delta").mode("overwrite").save(f"{location_bronze}/psgc_cities")